# Kinyarwanda news topic classification

Run this notebook from top to bottom in a fresh Google Colab runtime. It downloads the authors' real KINNEWS files, prepares the split, runs the linear and recurrent experiments, evaluates the selected classifier once on the held-out test set, and loads it for inference. The training cells may take several minutes.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo = Path("/content/kinyarwanda-news-classifier")
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/irachrist1/kinyarwanda-news-classifier.git", str(repo)], check=True)
os.chdir(repo)
print("Repository:", repo)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-training.txt"], check=True)


## Prepare the authors' data

The published split contains repeated articles. The preparation script creates a new stratified split after removing blanks, conflicting labels, and duplicates. It records source hashes and counts in `data/processed/manifest.json`.


In [ ]:
source = "https://drive.google.com/drive/folders/1zxn0hgrOLlUsK5V0c7l71eAj1t2jiyox"
if not (Path("data/raw/raw/train.csv").exists() and Path("data/raw/raw/test.csv").exists()):
    subprocess.run(["gdown", "--folder", source, "-O", "data/raw"], check=True)
subprocess.run([sys.executable, "-m", "src.data"], check=True)


## Compare one change at a time

Validation macro F1 is the selection metric because the 14 topics are imbalanced. Each row in the experiment log names its reference and one changed setting.


In [ ]:
for name in ["majority", "word_unigram", "word_bigram", "character", "word_character"]:
    subprocess.run([sys.executable, "-m", "src.linear", name], check=True)


In [ ]:
recurrent_runs = [
    ["gru"],
    ["lstm"],
    ["lstm", "--epochs", "8"],
    ["lstm", "--epochs", "8", "--sequence-length", "320"],
]
for arguments in recurrent_runs:
    subprocess.run([sys.executable, "-m", "src.recurrent", *arguments], check=True)


In [ ]:
import pandas as pd
subprocess.run([sys.executable, "-m", "src.experiment_log"], check=True)
pd.read_csv("results/experiment_log.csv")


## Evaluate the selected classifier

The evaluation module retrains the word-bigram baseline and the selected combined word and character classifier on training plus validation articles. It then scores the untouched test set, calculates paired uncertainty estimates, and saves predictions and a confusion matrix.


In [ ]:
subprocess.run([sys.executable, "-m", "src.evaluate"], check=True)
import json
metrics = json.loads(Path("results/test_metrics.json").read_text())
print("Baseline:", metrics["baseline"])
print("Selected:", metrics["selected"])
print("Uncertainty:", metrics["uncertainty"])


In [ ]:
import joblib
model = joblib.load("artifacts/final.joblib")
labels = json.loads(Path("artifacts/labels.json").read_text())
example = pd.read_csv("data/processed/test.csv").iloc[0]
prediction = int(model.predict([example["text"]])[0])
print("Predicted topic:", labels[str(prediction)]["kinyarwanda"])
print("Reference topic:", labels[str(int(example["label"]))]["kinyarwanda"])


The interactive application is in `app.py` and uses the same saved model. Run it locally with `streamlit run app.py`, or open the live link in the repository README when deployment is available.
